# Split-RAG experiments
This notebook implements the Split-RAG architecture, a specialized retrieval strategy designed to overcome the limitations identified during the baseline experiments. While the baseline Naive RAG treated legislation and jurisprudence as a single unstructured corpus, Split-RAG introduces a functional decoupling of these data sources.

**Baseline RAGAs results**: 
* faithfulness: 0.7205 
* factual correctness: 0.3340
* context precision: 0.8750
* context recall: 0.3417

## Imports

In [56]:
import os
import time
import pandas as pd
import json
from datasets import Dataset
from dotenv import load_dotenv

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_openai import AzureOpenAIEmbeddings
from langchain_chroma import Chroma
from azure.identity import DefaultAzureCredential, get_bearer_token_provider
from openai import AsyncAzureOpenAI
from langchain_classic.retrievers import ParentDocumentRetriever
from langchain_classic.retrievers import EnsembleRetriever
from langchain_classic.retrievers import BM25Retriever
from langchain_classic.retrievers import ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors import LLMChainFilter
from langchain_classic.storage import LocalFileStore, create_kv_docstore



from ragas.metrics import Faithfulness, FactualCorrectness, ContextPrecision, ContextRecall
from ragas.llms import llm_factory
import nest_asyncio
from ragas import aevaluate 
from ragas import RunConfig



C:\Users\verkad004\AppData\Local\Temp\ipykernel_26524\4256137953.py:23: DeprecationWarning: Importing Faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import Faithfulness
  from ragas.metrics import Faithfulness, FactualCorrectness, ContextPrecision, ContextRecall
C:\Users\verkad004\AppData\Local\Temp\ipykernel_26524\4256137953.py:23: DeprecationWarning: Importing FactualCorrectness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import FactualCorrectness
  from ragas.metrics import Faithfulness, FactualCorrectness, ContextPrecision, ContextRecall
C:\Users\verkad004\AppData\Local\Temp\ipykernel_26524\4256137953.py:23: DeprecationWarning: Importing ContextPrecision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' inst

## Environment variables

In [4]:
env_path = os.path.join("..", ".env")
load_dotenv(dotenv_path=env_path)

endpoint = os.getenv("AZURE_OPENAI_ENDPOINT")
api_version = os.getenv("AZURE_OPENAI_API_VERSION")
embedding_deployment = os.getenv("AZURE_EMBEDDING_DEPLOYMENT")
deployment = os.getenv("AZURE_OPENAI_DEPLOYMENT")

# Initialize Azure AD token provider
token_provider = get_bearer_token_provider(
    DefaultAzureCredential(),
    "https://cognitiveservices.azure.com/.default"
)

## Data Loading
In contrast to the baseline experiments, the Split-RAG architecture employs a decoupled data strategy. Rather than treating legislation and jurisprudence as a unified corpus, this approach maintains them as two distinct, specialized knowledge bases.

In [5]:
legislation_path = '../data/legislation_omgevingswet_cleaned.jsonl'
jurisprudence_path = '../data/jurisprudence_omgevingswet_cleaned.jsonl'

common_cols = ['id', 'title', 'text', 'word_count', 'source_type']
df_legislation = pd.read_json(legislation_path, lines=True)
df_jurisprudence = pd.read_json(jurisprudence_path, lines=True)

df_leg_sub = df_legislation[common_cols]
df_jur_sub = df_jurisprudence[common_cols]

print(f"Legislation loaded: {len(df_legislation)} rows")
print(f"Jurisprudence loaded: {len(df_jurisprudence)} rows")

Legislation loaded: 725 rows
Jurisprudence loaded: 3404 rows


In [6]:
# Legislation document creation
docs_legislation = [
    Document(
        page_content=str(row['text']),
        metadata={
            "id": row['id'],
            "title": row['title'],
            "source_type": row['source_type'],
            "word_count": row['word_count']
        }
    ) for _, row in df_leg_sub.iterrows()
]

# jurisprudence document creation
docs_jurisprudence = [
    Document(
        page_content=str(row['text']),
        metadata={
            "id": row['id'],
            "title": row['title'],
            "source_type": row['source_type'],
            "word_count": row['word_count']
        }
    ) for _, row in df_jur_sub.iterrows()
]

print(f"Created {len(docs_legislation)} legislation documents.")
print(f"Created {len(docs_jurisprudence)} jurisprudence documents.")

Created 725 legislation documents.
Created 3404 jurisprudence documents.


# Retrieving documents per dataset

In [7]:
embeddings = AzureOpenAIEmbeddings(
    azure_deployment=embedding_deployment, 
    azure_endpoint=endpoint,              
    openai_api_version=api_version,       
    azure_ad_token_provider=token_provider
)

## Legislation
For legislation, we are implementing a **Parent Document Retrieval (PDR)** system. Instead of standard chunking, this system employs a hierarchical approach.

**Rationale:**
Laws are inherently hierarchical and logically structured. Recent research (Lim et al., 2025; Guo et al., 2025) demonstrates that multi-granular context is superior to fixed chunk sizes. By indexing small child chunks (400 tokens), we maximise search precision. However, to prevent the AI from losing the legal context, the full parent” article is served to the model as context upon a match. This chunk-to-context reconstruction directly addresses the shortcomings of naive RAG systems when dealing with complex legislation (Lim et al., 2025).


In [24]:
# Splitter for search chunks (childs)
child_splitter = RecursiveCharacterTextSplitter(
    chunk_size=400, 
    chunk_overlap=50
)

# Define paths
vdb_base_path = "../data/vector_stores"
leg_child_path = os.path.join(vdb_base_path, "split_legislation_child")
leg_parent_path = os.path.join(vdb_base_path, "split_legislation_parent_store")

# Parent storage for disk
fs = LocalFileStore(leg_parent_path)
parent_docstore = create_kv_docstore(fs) 

# Child database
vdb_leg_child = Chroma(
    collection_name="leg_pdr_final",
    embedding_function=embeddings, 
    persist_directory=leg_child_path
)

# Retriever that links child chunks (cosine similarity)
retriever_leg_parent = ParentDocumentRetriever(
    vectorstore=vdb_leg_child,
    docstore=parent_docstore,
    child_splitter=child_splitter,
)

In [25]:
# Check if disk is empty and populate using unique, stable IDs
if vdb_leg_child._collection.count() == 0:
    print("Vectordatabase empty. Indexing legislation documents...")
    batch_size = 50
    
    for i in range(0, len(docs_legislation), batch_size):
        batch = docs_legislation[i : i + batch_size]
        
        # Extract fixed unique IDs from document metadata
        batch_ids = [str(doc.metadata["id"]) for doc in batch]
        
        # Populate parent docstore and child vector store simultaneously
        retriever_leg_parent.add_documents(batch, ids=batch_ids)
        
        print(f"Progress: {min(i + batch_size, len(docs_legislation))}/{len(docs_legislation)} articles done")
        time.sleep(1)
        
    # Force persistence to disk
    if hasattr(vdb_leg_child, 'persist'):
        vdb_leg_child.persist()
        
    print(f"Database is ready. Total chunks: {vdb_leg_child._collection.count()}")
else:
    print(f"Database is ready. Loaded {vdb_leg_child._collection.count()} chunks from disk.")

Database is ready. Loaded 4100 chunks from disk.


## Jurisprudence
For the jurisprudence, we use Structure-Aware Chunking.

**Rationale:**
Jurisprudence is narrative and story-driven in nature, meaning that relevant information is often scattered across lengthy texts. In line with the findings in *LegalBench-RAG* (Pipitone & Alami, 2024), standard retrieval is often insufficient for this type of document. We use a splitter that preserves the logical integrity of the judgment (splitting by ECLI and paragraphs). 

In [26]:
structure_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1500,
    chunk_overlap=150,
    length_function=len,
    separators=[
        "\nECLI:",          # Prioriteit voor de start van een nieuwe uitspraak
        "\n\n",            # Paragrafen
        "\n",              # Regels
        ". ",              # Zinnen
        " "                # Woorden
    ]
)

# Pas de splitter toe op je jurisprudentie documenten
chunks_juris_structure = structure_splitter.split_documents(docs_jurisprudence)

print(f"Jurisprudence: Structure-Aware Results")
print(f"Totaal aantal chunks: {len(chunks_juris_structure)}")

# Check de eerste chunk
if chunks_juris_structure:
    print(f"\nSample Metadata: {chunks_juris_structure[0].metadata}")
    print(f"Sample Content: {chunks_juris_structure[0].page_content[:300]}...")

Jurisprudence: Structure-Aware Results
Totaal aantal chunks: 51321

Sample Metadata: {'id': 'ECLI:NL:RBGEL:2024:26', 'title': 'ECLI:NL:RBGEL:2024:26, Rechtbank Gelderland, 05-01-2024, AWB-22_5249 en 22_5252', 'source_type': 'jurisprudence', 'word_count': 2403}
Sample Content: Weigering handhavingsverzoeken m.b.t. geitenhouderij. Intern salderen. Beroep gegrond vanwege een motiveringsgebrek. De rechtsgevolgen worden door de rechtbank in stand gelaten omdat in het verweerschrift afdoende is onderbouwd dat er geen sprake is van een overtreding van de Wnb.


    

Zittingspl...


In [27]:
def create_vdb_in_batches(chunks, path, name, embeddings, batch_size=50):
    print(f"Checking existing progress for: {name}")
    
    # 1. Open of maak de database aan
    vector_db = Chroma(
        persist_directory=path,
        embedding_function=embeddings
    )
    
    # 2. Kijk wat er al in zit (op basis van metadata 'id' of 'source')
    # We halen de unieke bronnen op die al verwerkt zijn
    existing_count = vector_db._collection.count()
    
    if existing_count > 0:
        print(f"Hervatten: Er staan al {existing_count} chunks in de database.")
        # We skippen de chunks die al aanwezig zijn
        # Let op: dit werkt het best als de volgorde van 'chunks' altijd hetzelfde is
        remaining_chunks = chunks[existing_count:]
    else:
        print("Geen bestaande data gevonden. We starten vanaf het begin.")
        remaining_chunks = chunks

    if not remaining_chunks:
        print(f"Alle chunks voor {name} zijn al verwerkt!\n")
        return vector_db

    # 3. Voeg de resterende chunks toe in batches
    print(f"Nog {len(remaining_chunks)} chunks toe te voegen...")
    
    for i in range(0, len(remaining_chunks), batch_size):
        batch = remaining_chunks[i : i + batch_size]
        vector_db.add_documents(documents=batch)
        
        current_total = existing_count + i + len(batch)
        print(f"Progress for {name}: {current_total}/{len(chunks)} chunks totaal...")
        
        # Voorkom rate limits en geef de schijf tijd om te schrijven
        time.sleep(1) 
        
    print(f"{name} Vector Store succesvol bijgewerkt bij {path}\n")
    return vector_db

# Pad voor de jurisprudentie database
juris_vdb_path = os.path.join(vdb_base_path, "split_jurisprudence_structure")

# Gebruik je vertrouwde functie om de vdb aan te maken
vdb_juris_split = create_vdb_in_batches(
    chunks=chunks_juris_structure, 
    path=juris_vdb_path, 
    name="Jurisprudence-Structure",
    embeddings=embeddings,
    batch_size=50
)

Checking existing progress for: Jurisprudence-Structure
Hervatten: Er staan al 51321 chunks in de database.
Alle chunks voor Jurisprudence-Structure zijn al verwerkt!



In [28]:
# Load database
vdb_juris_split = Chroma(
    persist_directory=os.path.join(vdb_base_path, "split_jurisprudence_structure"),
    embedding_function=embeddings
)

print(f"Jurisprudence chunks geladen: {vdb_juris_split._collection.count()}")

Jurisprudence chunks geladen: 51321


In [29]:
retriever_juris_structure = vdb_juris_split.as_retriever(
    search_type="mmr", 
    search_kwargs={
        "k": 15,                # fetch 15 fragments
        "fetch_k": 50,          # fetch 50 and retrieve 15 most diverse
        "lambda_mult": 0.5      # balance between relevance and diversity
    }
)

## Split-RAG

In [45]:
vdb_path = "../data/vector_stores"

vdb_leg_child = Chroma(
    persist_directory=os.path.join(vdb_path, "split_legislation_child"),
    embedding_function=embeddings
)

# Load parent store (complete articles)
fs_leg = LocalFileStore(os.path.join(vdb_path, "split_legislation_parent_store"))
parent_docstore = create_kv_docstore(fs_leg)

vdb_juris_split = Chroma(
    persist_directory=os.path.join(vdb_path, "split_jurisprudence_structure"),
    embedding_function=embeddings
)

# Check chunk counts for your Split-RAG architecture
print(f"Split-RAG Legislation chunks: {vdb_leg_child._collection.count()}")
print(f"Split-RAG Jurisprudence chunks: {vdb_juris_split._collection.count()}")

Split-RAG Legislation chunks: 2050
Split-RAG Jurisprudence chunks: 51321


## RAGAs evaluation

### QA pairs

In [31]:
# Load JSON file
file_path = "../data/QA_pairs_evaluation.json" 

with open(file_path, 'r', encoding='utf-8') as f:
    qa_list = json.load(f)

# Convert to DataFrame
df_qa = pd.DataFrame(qa_list)
print(f"Dataset geladen: {len(df_qa)} vragen gevonden.")

Dataset geladen: 10 vragen gevonden.


### RAGAs dataset generations

In [32]:
client = AsyncAzureOpenAI(
    azure_endpoint=endpoint,
    azure_deployment=deployment,
    api_version=api_version,
    azure_ad_token_provider=token_provider,
)

In [33]:
ensemble_retriever = EnsembleRetriever(
    retrievers=[retriever_leg_parent, retriever_juris_structure],
    weights=[0.6, 0.4] 
)

In [ ]:
async def run_evaluation_loop(retriever, dataset_list, strategy_name): 
    questions = []
    all_contexts = []
    ground_truths = []
    all_answers = []


    
    system_prompt = (
        "Je bent een ervaren juridisch adviseur voor de Gemeente Amsterdam, gespecialiseerd in de Omgevingswet. "
        "Beantwoord de vraag uitsluitend op basis van de verstrekte context. "
        "Formuleer een lopend, professioneel en volledig gemotiveerd juridisch antwoord van ongeveer 150 woorden."
        "Gebruik absoluut geen losse genummerde lijstjes of checklists en verweef alle vereiste details op een natuurlijke wijze in de lopende tekst.\n\n"
        "STRIKTE INHOUDELIJKE EISEN:\n"
        "- CONCLUSIE: Begin je antwoord direct in de allereerste zin met een helder en uitsluitsel gevend antwoord (bijvoorbeeld: 'Ja, ...' of 'Nee, ...') om de casus direct te beslechten.\n"
        "- RECHTSOMVORMING: De verstrekte context bevat vaak historische jurisprudentie gebaseerd op de oude Wabo of oude bestemmingsplannen. Vertaal deze logica proactief naar het stelsel van de Omgevingswet (bijv. een binnenplanse afwijking is nu een omgevingsplanactiviteit).\n"
        "- BRONVERMELDING: Integreer het specifieke wetsartikel uit de Omgevingswet en het relevante ECLI-nummer direct en vloeiend als onderdeel van de argumentatie.\n"
        "- VOORBEHOUD: Als cruciale informatie, wetsartikelen of ECLI-nummers ontbreken in de context om de vraag volledig te beantwoorden, verzin of hallucineer dan NOOIT gegevens, maar benoem dit voorbehoud op een professionele manier binnen de lopende tekst.\n\n"
        )


    print(f"Start retrieval and generation for: {strategy_name}...")

    for item in dataset_list:
        q = item['question']
        gt = item['ground_truth']
        
        # Retrieval
        docs = retriever.invoke(q) 
        ctx_list = [doc.page_content for doc in docs]
        context_text = "\n\n".join(ctx_list)
        
        # Generation
        resp = await client.chat.completions.create(
            model=deployment,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": f"Context: {context_text}\n\nVraag: {q}"},
            ],
            temperature=0 
        )
        answer = resp.choices[0].message.content
        
        # Save to lists
        questions.append(q)
        all_contexts.append(ctx_list) 
        ground_truths.append(gt)
        all_answers.append(answer)

    # Dataset object for RAGAs
    ds = Dataset.from_dict({
        "question": questions,
        "answer": all_answers,
        "contexts": all_contexts,
        "ground_truth": ground_truths
    })
    
    os.makedirs("../data/results", exist_ok=True)
    ds.to_pandas().to_csv(f"../data/results/results_{strategy_name}_2.csv", index=False, encoding='utf-16')
    
    return ds

dataset_split = await run_evaluation_loop(ensemble_retriever, qa_list, "split-rag")

Start retrieval and generation for: split-rag...


## Results and analysis

In [35]:
evaluator_llm = llm_factory(
    model=deployment,
    client=client,
    max_tokens=4096
    )

config = RunConfig(
    timeout=240,     
    max_retries=20, 
    max_wait=60,
    max_workers=1,
    seed=42,      
)

nest_asyncio.apply()

# Initialize metrics
metrics = [
    Faithfulness(llm=evaluator_llm),
    FactualCorrectness(llm=evaluator_llm), 
    ContextPrecision(llm=evaluator_llm),
    ContextRecall(llm=evaluator_llm)
]

# evaluate
result_split= await aevaluate(
    dataset=dataset_split,
    metrics=metrics,
    run_config=config
)


print(f"results split RAG: {result_split}")


C:\Users\verkad004\AppData\Local\Temp\ipykernel_26524\425720541.py:26: DeprecationWarning: aevaluate() is deprecated and will be removed in a future version. Use the @experiment decorator instead. See https://docs.ragas.io/en/latest/concepts/experiment/ for more information.
  result_split= await aevaluate(
Evaluating: 100%|██████████| 40/40 [14:01<00:00, 21.03s/it]

results split RAG: {'faithfulness': 0.8111, 'factual_correctness(mode=f1)': 0.3080, 'context_precision': 0.4942, 'context_recall': 0.6800}


# BM25 retriever + MMR
The initial results of the Split-RAG model show high faithfulness (0.85), but lower factual correctness (0.36) and context precision (0.46). This indicates embedding dilution: crucial legal identifiers, such as article numbers and ECLI references, become diluted in the semantic vector space (S & Easwarakumar, 2025).

To improve factual accuracy, a BM25Retriever has been added to the ensemble. Whereas the current vector search focuses on meaning, BM25 enforces precision based on exact keyword matches (lexical matching). This hybrid intervention is necessary to reduce noise in the context and restore the direct link between the query and the specific legal source.

In parallel, Maximum Marginal Relevance (MMR) filtering is introduced to optimize the jurisprudence retrieval channel. Because judicial rulings in Dutch administrative law are inherently narrative and lengthy, standard similarity searches frequently retrieve highly redundant context fragments that reiterate the same legal facts or definitions. This textual redundancy overloads the LLM's limited context window without introducing new information, further driving down context precision and confusing the generator. By implementing MMR, the pipeline actively penalizes redundant text segments and prioritizes information diversity. This ensures that the retrieved context contains a broader, more distinct range of legal perspectives and factual nuances necessary to formulate a comprehensive legal justification.

In [ ]:
# Branch 1: Legislation (PDR + BM25) 
# Fetch documents from vectorstore and handle string conversion safely
leg_data = vdb_leg_child.get()
raw_documents_leg = leg_data.get('documents', []) if leg_data else []

processed_texts_leg = [
    doc.page_content if hasattr(doc, 'page_content') else str(doc) 
    for doc in raw_documents_leg if doc is not None
]

if not processed_texts_leg:
    print("CRITICAL ERROR: 'processed_texts_leg' is empty!")
    print("Please verify that 'vdb_leg_child' is filled and properly initialized.")
    processed_texts_leg = ["Fallback context to prevent zero division error during testing."]

# Initialize sparse keyword retriever for legislation
retriever_leg_bm25 = BM25Retriever.from_texts(processed_texts_leg)
retriever_leg_bm25.k = 5

# Combine retrievers (0.7 dense PDR, 0.3 sparse BM25)
hybrid_leg = EnsembleRetriever(
    retrievers=[retriever_leg_parent, retriever_leg_bm25], 
    weights=[0.7, 0.3]
)

# Branch 2: Jurisprudence (MMR + BM25)
# Configure dense vector retriever to use MMR to reduce redundancy
retriever_juris_mmr = vdb_juris_split.as_retriever(
    search_type="mmr", 
    search_kwargs={"k": 15, "fetch_k": 50, "lambda_mult": 0.5}
)


# FIX: Fetch jurisprudence documents in batches to circumvent the SQLite "too many SQL variables" limit
processed_texts_juris = []
batch_limit = 10000
offset = 0

print("Fetching jurisprudence chunks in batches from disk...")
while True:
    juris_batch = vdb_juris_split.get(limit=batch_limit, offset=offset)
    raw_docs_juris_batch = juris_batch.get('documents', []) if 'juri_batch' in locals() else juris_batch.get('documents', [])
    
    if not raw_docs_juris_batch:
        break
        
    for doc in raw_docs_juris_batch:
        if doc is not None:
            processed_texts_juris.append(doc.page_content if hasattr(doc, 'page_content') else str(doc))
            
    print(f"Loaded database batch: chunks collected so far: {len(processed_texts_juris)}")
    
    if len(raw_docs_juris_batch) < batch_limit:
        break
    offset += batch_limit

# Explicit guard for the jurisprudence pipeline
if not processed_texts_juris:
    print("erro: 'processed_texts_juris' is empty!")
    print("Please verify that 'vdb_juris_split' is filled and properly initialized.")
    processed_texts_juris = ["Fallback context to prevent zero division error during testing."]

# Initialize sparse keyword retriever for jurisprudence
retriever_juris_bm25 = BM25Retriever.from_texts(processed_texts_juris)
retriever_juris_bm25.k = 10

# Combine retrievers using an equal 50/50 balance
hybrid_juris = EnsembleRetriever(
    retrievers=[retriever_juris_mmr, retriever_juris_bm25], 
    weights=[0.5, 0.5]
)

# Final hybrid ensemble (0.6 legislation, 0.4 jurisprudence)
hybrid_ensemble = EnsembleRetriever(
    retrievers=[hybrid_leg, hybrid_juris],
    weights=[0.6, 0.4]
)

# Use  Azure OpenAI LLM as the rerank compressor
llm_compressor = LLMChainFilter.from_llm(client)

# Wrap the ensemble retriever with the LLM compressor
reranked_retriever = ContextualCompressionRetriever(
    base_compressor=llm_compressor, 
    base_retriever=hybrid_ensemble
)

print("run 6 complete!")

Fetching jurisprudence chunks in batches from disk...
Loaded database batch: chunks collected so far: 10000
Loaded database batch: chunks collected so far: 20000
Loaded database batch: chunks collected so far: 30000
Loaded database batch: chunks collected so far: 40000
Loaded database batch: chunks collected so far: 50000
Loaded database batch: chunks collected so far: 51321
run 3 complete!


In [ ]:
dataset_rerank = await run_evaluation_loop(reranked_retriever, qa_list, "hybrid-split-rag-rerank")

result_hybrid= await aevaluate(
    dataset=dataset_rerank,
    metrics=metrics,
    run_config=config
)

print(f"results split RAG rerank: {result_hybrid}")

Start retrieval and generation for: hybrid-split-rag_3...


C:\Users\verkad004\AppData\Local\Temp\ipykernel_26524\1718964530.py:4: DeprecationWarning: aevaluate() is deprecated and will be removed in a future version. Use the @experiment decorator instead. See https://docs.ragas.io/en/latest/concepts/experiment/ for more information.
  result_hybrid= await aevaluate(
Evaluating:  60%|██████    | 24/40 [15:24<10:25, 39.11s/it]Exception raised in Job[24]: TimeoutError()
AzureCliCredential.get_token_info failed: Failed to invoke the Azure CLI
Evaluating: 100%|██████████| 40/40 [3:58:28<00:00, 357.72s/it]    

results split RAG hybrid: {'faithfulness': 0.7743, 'factual_correctness(mode=f1)': 0.2622, 'context_precision': 0.3728, 'context_recall': 0.6717}


In [ ]:
# Load split-rag data
df_split = pd.read_csv('../data/results/results_hybrid-split-rag-rerank.csv',encoding='utf-16')


# print split-rag data
for i in range(len(df_split)):
    print(f"Case {i+1}\n")
    print(f"Question:\n{df_split.loc[i, 'question']}\n")
    
    print(f"Answer:\n")
    print(f"{df_split.loc[i, 'answer']}")
    print("-" * 50)
    

Case 1

Question:
Kan een omgevingsvergunning voor een dakterras op een gemeentelijk monument worden verleend als het hekwerk de maximale bouwhoogte overschrijdt?

Answer:

Nee, een omgevingsvergunning voor een dakterras op een gemeentelijk monument kan niet worden verleend als het hekwerk de maximale bouwhoogte overschrijdt, tenzij er sprake is van een expliciete afwijking die voldoet aan de voorwaarden van de Omgevingswet en de relevante gemeentelijke erfgoedverordening. Op grond van artikel 16.15a van de Omgevingswet dient bij een aanvraag voor een omgevingsvergunning voor een gemeentelijk monument de gemeentelijke erfgoedcommissie als adviseur te worden betrokken. Daarnaast bepaalt artikel 16.58 dat bij een rijksmonument overleg met de eigenaar verplicht is, en dat bij wezenlijke belangen van monumentenzorg instemming vereist kan zijn. Hoewel dit artikel niet direct van toepassing is op gemeentelijke monumenten, biedt het een vergelijkbaar toetsingskader.

In de context van een dak